---

## load swe_benchmark_data , using lite ( smaller datasets ) as starters

In [1]:
from datasets import load_dataset
swe_bench_data = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")


/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 300/300 [00:00<00:00, 18463.01 examples/s]


In [2]:
sample=swe_bench_data[0]
sample

{'repo': 'astropy/astropy',
 'instance_id': 'astropy__astropy-12907',
 'base_commit': 'd16bfe05a744909de4b27f5875fe0d4ed41ce607',
 'patch': "diff --git a/astropy/modeling/separable.py b/astropy/modeling/separable.py\n--- a/astropy/modeling/separable.py\n+++ b/astropy/modeling/separable.py\n@@ -242,7 +242,7 @@ def _cstack(left, right):\n         cright = _coord_matrix(right, 'right', noutp)\n     else:\n         cright = np.zeros((noutp, right.shape[1]))\n-        cright[-right.shape[0]:, -right.shape[1]:] = 1\n+        cright[-right.shape[0]:, -right.shape[1]:] = right\n \n     return np.hstack([cleft, cright])\n \n",
 'test_patch': "diff --git a/astropy/modeling/tests/test_separable.py b/astropy/modeling/tests/test_separable.py\n--- a/astropy/modeling/tests/test_separable.py\n+++ b/astropy/modeling/tests/test_separable.py\n@@ -28,6 +28,13 @@\n p1 = models.Polynomial1D(1, name='p1')\n \n \n+cm_4d_expected = (np.array([False, False, True, True]),\n+                  np.array([[True,  Tr

In [3]:
from agentless.fl.localize import localize_instance
start_file_locs = None
existing_instance_ids = set()


In [4]:
from agentless.util.preprocess_data import (
    check_contains_valid_loc,
    filter_none_python,
    filter_out_test_files,
    get_repo_structure,
)

---- 
cloneing and renaming the repository per sample data indicated

In [5]:
from get_repo_structure.get_repo_structure import (
    get_project_structure_from_scratch,
    parse_python_file,
)
d = get_project_structure_from_scratch(
            sample["repo"], sample["base_commit"], sample["instance_id"], "playground"
        )
structure = d["structure"]


Cloning repository from https://github.com/astropy/astropy.git to playground/d1cd9c3e-72e3-4937-bc31-cc094e8c3c77/astropy...


Cloning into 'playground/d1cd9c3e-72e3-4937-bc31-cc094e8c3c77/astropy'...


Repository cloned successfully.
Checking out commit d16bfe05a744909de4b27f5875fe0d4ed41ce607 in repository at playground/d1cd9c3e-72e3-4937-bc31-cc094e8c3c77/astropy...


Note: switching to 'd16bfe05a744909de4b27f5875fe0d4ed41ce607'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at d16bfe05a7 Merge pull request #12900 from Cadair/custom_compound_model


Commit checked out successfully.


In [6]:


structure.keys()

dict_keys(['astropy', 'licenses', 'cextern', '.github', '.pyinstaller', '.circleci', '.git', 'examples', 'docs'])

---- 

## check if all the prompts works with NVIDIA NIM llama3 405B model 

### 1) Obtain top N files :   **obtain_relevant_files_prompt**

In [8]:

    obtain_irrelevant_files_prompt = """
Please look through the following GitHub problem description and Repository structure and provide a list of folders that are irrelevant to fixing the problem.
Note that irrelevant folders are those that do not need to be modified and are safe to ignored when trying to solve this problem.

### GitHub Problem Description ###
{problem_statement}

###

### Repository Structure ###
{structure}

###

Please only provide the full path.
Remember that any subfolders will be considered as irrelevant if you provide the parent folder.
Please ensure that the provided irrelevant folders do not include any important files needed to fix the problem
The returned folders should be separated by new lines and wrapped with ```
For example:
```
folder1/
folder2/folder3/
folder4/folder5/
```
"""

    file_content_template = """
### File: {file_name} ###
{file_content}
"""
    file_content_in_block_template = """
### File: {file_name} ###
```python
{file_content}
```
"""

    obtain_relevant_code_combine_top_n_prompt = """
Please review the following GitHub problem description and relevant files, and provide a set of locations that need to be edited to fix the issue.
The locations can be specified as class names, function or method names, or exact line numbers that require modification.

### GitHub Problem Description ###
{problem_statement}

###
{file_contents}

###

Please provide the class name, function or method name, or the exact line numbers that need to be edited.
The possible location outputs should be either "class", "function" or "line".

### Examples:
```
full_path1/file1.py
line: 10
class: MyClass1
line: 51

full_path2/file2.py
function: MyClass2.my_method
line: 12

full_path3/file3.py
function: my_function
line: 24
line: 156
```

Return just the location(s) wrapped with ```.
"""

    obtain_relevant_code_combine_top_n_no_line_number_prompt = """
Please review the following GitHub problem description and relevant files, and provide a set of locations that need to be edited to fix the issue.
The locations can be specified as class, method, or function names that require modification.

### GitHub Problem Description ###
{problem_statement}

###
{file_contents}

###

Please provide the class, method, or function names that need to be edited.
### Examples:
```
full_path1/file1.py
function: my_function1
class: MyClass1

full_path2/file2.py
function: MyClass2.my_method
class: MyClass3

full_path3/file3.py
function: my_function2
```

Return just the location(s) wrapped with ```.
"""
obtain_relevant_functions_and_vars_from_compressed_files_prompt_more = """
Please look through the following GitHub Problem Description and the Skeleton of Relevant Files.
Identify all locations that need inspection or editing to fix the problem, including directly related areas as well as any potentially related global variables, functions, and classes.
For each location you provide, either give the name of the class, the name of a method in a class, the name of a function, or the name of a global variable.

### GitHub Problem Description ###
{problem_statement}

### Skeleton of Relevant Files ###
{file_contents}

###

Please provide the complete set of locations as either a class name, a function name, or a variable name.
Note that if you include a class, you do not need to list its specific methods.
You can include either the entire class or don't include the class name and instead include specific methods in the class.
### Examples:
```
full_path1/file1.py
function: my_function_1
class: MyClass1
function: MyClass2.my_method

full_path2/file2.py
variable: my_var
function: MyClass3.my_method

full_path3/file3.py
function: my_function_2
function: my_function_3
function: MyClass4.my_method_1
class: MyClass5
```

Return just the locations wrapped with ```.
"""

    obtain_relevant_functions_and_vars_from_raw_files_prompt = """
Please look through the following GitHub Problem Description and Relevant Files.
Identify all locations that need inspection or editing to fix the problem, including directly related areas as well as any potentially related global variables, functions, and classes.
For each location you provide, either give the name of the class, the name of a method in a class, the name of a function, or the name of a global variable.

### GitHub Problem Description ###
{problem_statement}

### Relevant Files ###
{file_contents}

###

Please provide the complete set of locations as either a class name, a function name, or a variable name.
Note that if you include a class, you do not need to list its specific methods.
You can include either the entire class or don't include the class name and instead include specific methods in the class.
### Examples:
```
full_path1/file1.py
function: my_function_1
class: MyClass1
function: MyClass2.my_method

full_path2/file2.py
variable: my_var
function: MyClass3.my_method

full_path3/file3.py
function: my_function_2
function: my_function_3
function: MyClass4.my_method_1
class: MyClass5
```

Return just the locations wrapped with ```.
"""


In [12]:
from agentless.util.model import make_model
from agentless.util.model import NVIDIAChatDecoder
from agentless.util.utils import load_existing_instance_ids, load_jsonl, setup_logger
model="nvdev/meta/llama-3.1-405b-instruct"
backend ='nvidia'
logger = setup_logger('logger.log')
model = make_model(
            model=model,
            backend=backend,
            logger=logger,
            max_tokens=4096,
            temperature=0,
            batch_size=1,
        )

In [15]:
from agentless.util.preprocess_data import (
    correct_file_paths,
    get_full_file_paths_and_classes_and_functions,
    get_repo_files,
    line_wrap_content,
    show_project_structure,
)
d_to_str=show_project_structure(structure) ## format the dictionary repo structure to string
d_to_str

'astropy/\n    setup.py\n    conftest.py\n    logger.py\n    __init__.py\n    version.py\n    wcs/\n        wcslint.py\n        docstrings.py\n        __init__.py\n        utils.py\n        wcs.py\n        setup_package.py\n        wcsapi/\n            high_level_wcs_wrapper.py\n            high_level_api.py\n            sliced_low_level_wcs.py\n            __init__.py\n            utils.py\n            fitswcs.py\n            low_level_api.py\n            conftest.py\n            wrappers/\n                base.py\n                sliced_wcs.py\n                __init__.py\n    timeseries/\n        sampled.py\n        downsample.py\n        __init__.py\n        binned.py\n        core.py\n        io/\n            __init__.py\n            kepler.py\n        periodograms/\n            base.py\n            __init__.py\n            bls/\n                __init__.py\n                setup_package.py\n                methods.py\n                core.py\n            lombscargle/\n           

In [ ]:
from agentless.util.preprocess_data import (
    check_contains_valid_loc,
    filter_none_python,
    filter_out_test_files,
    get_repo_structure,
)

structure = d["structure"]
instance_id=sample["instance_id"]

#logger.info(f"================ localize {instance_id} ================")

bench_data = [x for x in swe_bench_data if x["instance_id"] == instance_id][0]
problem_statement = bench_data["problem_statement"]

filter_none_python(structure)  # some basic filtering steps
filter_out_test_files(structure)


In [17]:
obtain_relevant_files_prompt = """
Please look through the following GitHub problem description and Repository structure and provide a list of files that one would need to edit to fix the problem.

### GitHub Problem Description ###
{problem_statement}

###

### Repository Structure ###
{structure}

###

Please only provide the full path and return at most 5 files.
The returned files should be separated by new lines ordered by most to least important and wrapped with ```
For example:
```
file1.py
file2.py
```
"""

message = obtain_relevant_files_prompt.format(
            problem_statement=problem_statement,
            structure=d_to_str.strip(),
        ).strip()

In [18]:
message

"Please look through the following GitHub problem description and Repository structure and provide a list of files that one would need to edit to fix the problem.\n\n### GitHub Problem Description ###\nModeling's `separability_matrix` does not compute separability correctly for nested CompoundModels\nConsider the following model:\r\n\r\n```python\r\nfrom astropy.modeling import models as m\r\nfrom astropy.modeling.separable import separability_matrix\r\n\r\ncm = m.Linear1D(10) & m.Linear1D(5)\r\n```\r\n\r\nIt's separability matrix as you might expect is a diagonal:\r\n\r\n```python\r\n>>> separability_matrix(cm)\r\narray([[ True, False],\r\n       [False,  True]])\r\n```\r\n\r\nIf I make the model more complex:\r\n```python\r\n>>> separability_matrix(m.Pix2Sky_TAN() & m.Linear1D(10) & m.Linear1D(5))\r\narray([[ True,  True, False, False],\r\n       [ True,  True, False, False],\r\n       [False, False,  True, False],\r\n       [False, False, False,  True]])\r\n```\r\n\r\nThe output mat

In [19]:
traj = model.codegen(message, num_samples=1)[0]


In [24]:
traj

{'response': 'Based on the problem description, I would recommend editing the following files to fix the issue:\n\n\nastropy/modeling/separable.py\nastropy/modeling/core.py\nastropy/modeling/models.py\nastropy/modeling/parameters.py\nastropy/modeling/utils.py\n\n\nThese files are likely to contain the relevant code for computing separability matrices for nested CompoundModels. The `separable.py` file is the most important one to edit, as it contains the `separability_matrix` function that is causing the issue. The other files may also need to be modified to ensure that the separability matrix is correctly computed for nested CompoundModels.',
 'usage': {'completion_tokens': 130, 'prompt_tokens': 3171}}

In [23]:
raw_output=traj['response']
def _parse_model_return_lines( content: str) -> list[str]:
        if content:
            return content.strip().split("\n")
model_found_files=_parse_model_return_lines(raw_output)
model_found_files = [ f for f in model_found_files if f not in [''] ]
model_found_files

['Based on the problem description, I would recommend editing the following files to fix the issue:',
 'astropy/modeling/separable.py',
 'astropy/modeling/core.py',
 'astropy/modeling/models.py',
 'astropy/modeling/parameters.py',
 'astropy/modeling/utils.py',
 'These files are likely to contain the relevant code for computing separability matrices for nested CompoundModels. The `separable.py` file is the most important one to edit, as it contains the `separability_matrix` function that is causing the issue. The other files may also need to be modified to ensure that the separability matrix is correctly computed for nested CompoundModels.']

In [28]:
from agentless.util.preprocess_data import (
    correct_file_paths,
    get_full_file_paths_and_classes_and_functions,
    get_repo_files,
    line_wrap_content,
    show_project_structure,
)
file_contents = get_repo_files(structure,filepaths=model_found_files[1:-1])

In [32]:
print(file_contents.keys())
print(" >>> the below is the located file of python script from **astropy/modeling/separable.py**", '\n\n') 
file_contents['astropy/modeling/separable.py']

dict_keys(['astropy/modeling/separable.py', 'astropy/modeling/core.py', 'astropy/modeling/models.py', 'astropy/modeling/parameters.py', 'astropy/modeling/utils.py'])
 >>> the below is the located file of python script from **astropy/modeling/separable.py** 




'# Licensed under a 3-clause BSD style license - see LICENSE.rst\n\n"""\nFunctions to determine if a model is separable, i.e.\nif the model outputs are independent.\n\nIt analyzes ``n_inputs``, ``n_outputs`` and the operators\nin a compound model by stepping through the transforms\nand creating a ``coord_matrix`` of shape (``n_outputs``, ``n_inputs``).\n\n\nEach modeling operator is represented by a function which\ntakes two simple models (or two ``coord_matrix`` arrays) and\nreturns an array of shape (``n_outputs``, ``n_inputs``).\n\n"""\n\nimport numpy as np\n\nfrom .core import Model, ModelDefinitionError, CompoundModel\nfrom .mappings import Mapping\n\n\n__all__ = ["is_separable", "separability_matrix"]\n\n\ndef is_separable(transform):\n    """\n    A separability test for the outputs of a transform.\n\n    Parameters\n    ----------\n    transform : `~astropy.modeling.core.Model`\n        A (compound) model.\n\n    Returns\n    -------\n    is_separable : ndarray\n        A boole

In [37]:
from agentless.util.compress_file import CompressTransformer
from colorama import Fore
compress_assign = True
total_lines=30,
prefix_lines=10
suffix_lines=10


def _compress_assign_stmts(raw_code, total_lines=30, prefix_lines=10, suffix_lines=10):
    try:
        tree = cst.parse_module(raw_code)
    except Exception as e:
        print(e.__class__.__name__, e)
        return raw_code

    wrapper = cst.metadata.MetadataWrapper(tree)
    visitor = GlobalVariableVisitor()
    wrapper.visit(visitor)

    remove_line_intervals = []
    for stmt in visitor.assigns:
        print(Fore.BLUE , stmt)
        if stmt[2].line - stmt[1].line > total_lines:
            remove_line_intervals.append(
                (stmt[1].line + prefix_lines, stmt[2].line - suffix_lines)
            )
    return remove_lines(raw_code, remove_line_intervals)



def _get_skeleton(
    raw_code,
    keep_constant: bool = True,
    keep_indent: bool = False,
    compress_assign: bool = False,
    total_lines=30,
    prefix_lines=10,
    suffix_lines=10,
):
    try:
        tree = cst.parse_module(raw_code)
    except:
        return raw_code

    transformer = CompressTransformer(keep_constant=keep_constant, keep_indent=True)
    modified_tree = tree.visit(transformer)
    code = modified_tree.code

    if compress_assign:
        code = _compress_assign_stmts(
            code,
            total_lines=total_lines,
            prefix_lines=prefix_lines,
            suffix_lines=suffix_lines,
        )

    if keep_indent:
        code = code.replace(CompressTransformer.replacement_string + "\n", "...\n")
        code = code.replace(CompressTransformer.replacement_string, "...\n")
    else:
        pattern = f"\\n[ \\t]*{CompressTransformer.replacement_string}"
        replacement = "\n..."
        code = re.sub(pattern, replacement, code)

    return code


compressed_file_contents = {
            fn: _get_skeleton(
                code,
                compress_assign=compress_assign,
                total_lines=total_lines,
                prefix_lines=prefix_lines,
                suffix_lines=suffix_lines,
            )
            for fn, code in file_contents.items()
        }

In [39]:
compressed_file_contents

{'astropy/modeling/separable.py': '# Licensed under a 3-clause BSD style license - see LICENSE.rst\n\n"""\nFunctions to determine if a model is separable, i.e.\nif the model outputs are independent.\n\nIt analyzes ``n_inputs``, ``n_outputs`` and the operators\nin a compound model by stepping through the transforms\nand creating a ``coord_matrix`` of shape (``n_outputs``, ``n_inputs``).\n\n\nEach modeling operator is represented by a function which\ntakes two simple models (or two ``coord_matrix`` arrays) and\nreturns an array of shape (``n_outputs``, ``n_inputs``).\n\n"""\n\nimport numpy as np\n\nfrom .core import Model, ModelDefinitionError, CompoundModel\nfrom .mappings import Mapping\n\n\n__all__ = ["is_separable", "separability_matrix"]\n\n\ndef is_separable(transform):\n    """\n    A separability test for the outputs of a transform.\n\n    Parameters\n    ----------\n    transform : `~astropy.modeling.core.Model`\n        A (compound) model.\n\n    Returns\n    -------\n    is_se

---
 ## obtain classes and functions : **obtain_relevant_functions_and_vars_from_compressed_files_prompt_more **

In [ ]:
obtain_relevant_functions_and_vars_from_compressed_files_prompt_more = """
Please look through the following GitHub Problem Description and the Skeleton of Relevant Files.
Identify all locations that need inspection or editing to fix the problem, including directly related areas as well as any potentially related global variables, functions, and classes.
For each location you provide, either give the name of the class, the name of a method in a class, the name of a function, or the name of a global variable.

### GitHub Problem Description ###
{problem_statement}

### Skeleton of Relevant Files ###
{file_contents}

###

Please provide the complete set of locations as either a class name, a function name, or a variable name.
Note that if you include a class, you do not need to list its specific methods.
You can include either the entire class or don't include the class name and instead include specific methods in the class.
### Examples:
```
full_path1/file1.py
function: my_function_1
class: MyClass1
function: MyClass2.my_method

full_path2/file2.py
variable: my_var
function: MyClass3.my_method

full_path3/file3.py
function: my_function_2
function: my_function_3
function: MyClass4.my_method_1
class: MyClass5
```

Return just the locations wrapped with ```.
"""


In [9]:
from agentless.fl.FL import LLMFL
from agentless.util.utils import load_existing_instance_ids, load_jsonl, setup_logger

logger = setup_logger('logger.log')

found_files = []
found_related_locs = {}
found_edit_locs = {}
additional_artifact_loc_file = None
additional_artifact_loc_related = None
additional_artifact_loc_edit_location = None
file_traj, related_loc_trajs, edit_loc_traj = {}, [], {}

file_level=True

fl=LLMFL(instance_id,structure,problem_statement, "nvdev/meta/llama-3.1-405b-instruct", 'nvidia', logger)
found_files, additional_artifact_loc_file, file_traj = fl.localize_irrelevant( mock=False) # set mock=False, so we are really running the localization job
print(found_files)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
print(additional_artifact_loc_file)
print("---*10")
print("\n\n\n")
print(file_traj)